In [1]:
# Cell 1. Final pre-sleep inference artifact audit
# 원하는 결과:
# - 최종 대표 모델의 model/scaler/imputer/feature list/metadata 경로를 확인한다.
# - inference에 필요한 artifact가 모두 존재하는지 점검한다.
# - 최종 모델 checkpoint 내부 정보를 확인한다.
# - 아직 새 예측 함수는 만들지 않는다.

from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib
import torch

PROJECT_ROOT = Path(r"C:\workSpace\DeepLearnin_sleep")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
LOG_PATH = PROJECT_ROOT / "log" / "2026-06-29.md"

STAGE1_DIR = PROCESSED_DIR / "pre_sleep_forecasting" / "design_c_stage1"
STAGE1_TENSOR_DIR = STAGE1_DIR / "mlp_current_day_final"

METADATA_PATH = STAGE1_DIR / "metadata.json"
FEATURE_COLUMNS_PATH = STAGE1_DIR / "pre_sleep_design_c_stage1_final_feature_columns.csv"
IMPUTER_PATH = STAGE1_DIR / "pre_sleep_design_c_stage1_median_imputer.joblib"
SCALER_PATH = STAGE1_DIR / "pre_sleep_design_c_stage1_standard_scaler.joblib"

STABILITY_OUTPUT_DIR = STAGE1_DIR / "experiments" / "stage1_hyperparameter_stability_outputs"
SELECTION_SUMMARY_PATH = STABILITY_OUTPUT_DIR / "stage1_best_config_seed_selection_summary.csv"
THRESHOLD_POLICY_PATH = STABILITY_OUTPUT_DIR / "stage1_selected_model_threshold_policy_comparison.csv"

INFERENCE_PACKAGE_DIR = STAGE1_DIR / "inference_package"
INFERENCE_PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
feature_columns_df = pd.read_csv(FEATURE_COLUMNS_PATH, encoding="utf-8-sig")
selection_summary_df = pd.read_csv(SELECTION_SUMMARY_PATH, encoding="utf-8-sig")
threshold_policy_df = pd.read_csv(THRESHOLD_POLICY_PATH, encoding="utf-8-sig")

selected_row = selection_summary_df[
    selection_summary_df["selection_role"] == "selected_by_validation"
].iloc[0]

SELECTED_EXPERIMENT_ID = selected_row["experiment_id"]
SELECTED_THRESHOLD = float(selected_row["selected_threshold_from_validation"])
MODEL_PATH = PROJECT_ROOT / selected_row["model_path"]

artifact_paths = {
    "metadata": METADATA_PATH,
    "feature_columns": FEATURE_COLUMNS_PATH,
    "imputer": IMPUTER_PATH,
    "scaler": SCALER_PATH,
    "selected_model": MODEL_PATH,
    "selection_summary": SELECTION_SUMMARY_PATH,
    "threshold_policy": THRESHOLD_POLICY_PATH,
}

artifact_check_df = pd.DataFrame(
    [
        {
            "artifact": name,
            "path": str(path.relative_to(PROJECT_ROOT)),
            "exists": path.exists(),
            "size_kb": path.stat().st_size / 1024 if path.exists() else np.nan,
        }
        for name, path in artifact_paths.items()
    ]
)

checkpoint = torch.load(MODEL_PATH, map_location="cpu")
imputer = joblib.load(IMPUTER_PATH)
scaler = joblib.load(SCALER_PATH)

checkpoint_summary = pd.DataFrame(
    [
        {"metric": "selected_experiment_id", "value": SELECTED_EXPERIMENT_ID},
        {"metric": "selected_threshold", "value": SELECTED_THRESHOLD},
        {"metric": "checkpoint_experiment_id", "value": checkpoint.get("experiment_id")},
        {"metric": "checkpoint_input_dim", "value": checkpoint.get("input_dim")},
        {"metric": "checkpoint_hidden_dims", "value": str(checkpoint.get("hidden_dims"))},
        {"metric": "checkpoint_dropout", "value": checkpoint.get("dropout")},
        {"metric": "checkpoint_best_epoch", "value": checkpoint.get("best_epoch")},
        {
            "metric": "checkpoint_selected_threshold_from_validation",
            "value": checkpoint.get("selected_threshold_from_validation"),
        },
        {"metric": "feature_columns_csv_count", "value": len(feature_columns_df)},
        {"metric": "checkpoint_feature_count", "value": len(checkpoint.get("feature_columns", []))},
        {"metric": "imputer_features", "value": getattr(imputer, "n_features_in_", None)},
        {"metric": "scaler_features", "value": getattr(scaler, "n_features_in_", None)},
    ]
)

csv_features = feature_columns_df["feature"].astype(str).tolist()
checkpoint_features = [str(feature) for feature in checkpoint.get("feature_columns", [])]

feature_alignment_checks = pd.DataFrame(
    [
        {
            "check": "csv_vs_checkpoint_same_order",
            "passed": csv_features == checkpoint_features,
        },
        {
            "check": "csv_count_matches_checkpoint_input_dim",
            "passed": len(csv_features) == checkpoint.get("input_dim"),
        },
        {
            "check": "imputer_count_matches_csv",
            "passed": getattr(imputer, "n_features_in_", None) == len(csv_features),
        },
        {
            "check": "scaler_count_matches_csv",
            "passed": getattr(scaler, "n_features_in_", None) == len(csv_features),
        },
    ]
)

official_policy_test_row = threshold_policy_df[
    (threshold_policy_df["threshold_policy"] == "validation_balanced_accuracy")
    & (threshold_policy_df["evaluation_split"] == "test")
].iloc[0]

recall_policy_test_row = threshold_policy_df[
    (threshold_policy_df["threshold_policy"] == "validation_recall_priority_precision_at_least_0_50")
    & (threshold_policy_df["evaluation_split"] == "test")
].iloc[0]

threshold_policy_summary_df = pd.DataFrame(
    [
        {
            "policy": "official_validation_balanced_accuracy",
            "threshold": official_policy_test_row["selected_threshold"],
            "test_balanced_accuracy": official_policy_test_row["balanced_accuracy"],
            "test_precision": official_policy_test_row["precision"],
            "test_recall": official_policy_test_row["recall"],
        },
        {
            "policy": "optional_validation_recall_priority",
            "threshold": recall_policy_test_row["selected_threshold"],
            "test_balanced_accuracy": recall_policy_test_row["balanced_accuracy"],
            "test_precision": recall_policy_test_row["precision"],
            "test_recall": recall_policy_test_row["recall"],
        },
    ]
)

print("=== Final Inference Artifact Audit ===")
print("selected experiment:", SELECTED_EXPERIMENT_ID)
print("selected threshold:", SELECTED_THRESHOLD)
print("inference package dir:", INFERENCE_PACKAGE_DIR.relative_to(PROJECT_ROOT))

print("\n=== Artifact Check ===")
display(artifact_check_df)

print("\n=== Checkpoint / Preprocessing Summary ===")
display(checkpoint_summary)

print("\n=== Feature Alignment Checks ===")
display(feature_alignment_checks)

print("\n=== Feature Group Counts ===")
display(feature_columns_df["feature_group"].value_counts().reset_index(name="features"))

print("\n=== Threshold Policy Summary ===")
display(threshold_policy_summary_df)

print("\n=== Validation Checks ===")
print("missing artifacts:", int((~artifact_check_df["exists"]).sum()))
print("failed alignment checks:", int((~feature_alignment_checks["passed"]).sum()))

if artifact_check_df["exists"].all() and feature_alignment_checks["passed"].all():
    print("inference artifact audit passed")
else:
    print("inference artifact audit failed; inspect before packaging")

=== Final Inference Artifact Audit ===
selected experiment: presleep_stage1_hp_tiny_dropout40_wd1e3_seed2027
selected threshold: 0.54
inference package dir: data\processed\pre_sleep_forecasting\design_c_stage1\inference_package

=== Artifact Check ===


,artifact,path,exists,size_kb
0,metadata,data\processed\pre_sleep_forecasting\design_c_...,True,15.056641
1,feature_columns,data\processed\pre_sleep_forecasting\design_c_...,True,3.082031
2,imputer,data\processed\pre_sleep_forecasting\design_c_...,True,3.866211
3,scaler,data\processed\pre_sleep_forecasting\design_c_...,True,2.241211
4,selected_model,data\processed\pre_sleep_forecasting\design_c_...,True,17.288086
5,selection_summary,data\processed\pre_sleep_forecasting\design_c_...,True,2.224609
6,threshold_policy,data\processed\pre_sleep_forecasting\design_c_...,True,1.851562



=== Checkpoint / Preprocessing Summary ===


,metric,value
0,selected_experiment_id,presleep_stage1_hp_tiny_dropout40_wd1e3_seed2027
1,selected_threshold,0.54
2,checkpoint_experiment_id,presleep_stage1_hp_tiny_dropout40_wd1e3_seed2027
3,checkpoint_input_dim,58
4,checkpoint_hidden_dims,"(24, 12)"
5,checkpoint_dropout,0.4
6,checkpoint_best_epoch,12
7,checkpoint_selected_threshold_from_validation,0.54
8,feature_columns_csv_count,58
9,checkpoint_feature_count,58



=== Feature Alignment Checks ===


...


=== Feature Group Counts ===


,feature_group,features
0,missing_indicator,25
1,pre_sleep_intraday,18
2,previous_day_daily,9
3,timing,6



=== Threshold Policy Summary ===


,policy,threshold,test_balanced_accuracy,test_precision,test_recall
0,official_validation_balanced_accuracy,0.54,0.649209,0.65534,0.424528
1,optional_validation_recall_priority,0.47,0.659165,0.56000,0.572327



=== Validation Checks ===
missing artifacts: 0
failed alignment checks: 2
inference artifact audit failed; inspect before packaging


In [2]:
# Cell 2. Build corrected inference manifest with 70-to-58 preprocessing contract
# 원하는 결과:
# - inference 입력은 Stage 1 원본 70개 feature임을 명시한다.
# - imputer/scaler는 70개 feature 기준으로 적용하고, 이후 zero-variance feature를 제거해 58개 model input을 만든다.
# - 70개 input feature, 제거 feature 12개, 최종 model feature 58개의 정렬을 검증한다.
# - inference manifest를 저장하고 log를 업데이트한다.

RAW_FEATURE_COLUMNS_PATH = STAGE1_DIR / "pre_sleep_design_c_stage1_feature_columns.csv"
ZERO_VARIANCE_FEATURES_PATH = STAGE1_DIR / "pre_sleep_design_c_stage1_zero_variance_removed_features.csv"
FINAL_FEATURE_COLUMNS_PATH = STAGE1_DIR / "pre_sleep_design_c_stage1_final_feature_columns.csv"

INFERENCE_MANIFEST_PATH = INFERENCE_PACKAGE_DIR / "pre_sleep_inference_manifest.json"
INFERENCE_FEATURE_CONTRACT_PATH = INFERENCE_PACKAGE_DIR / "pre_sleep_inference_feature_contract.csv"

raw_feature_columns_df = pd.read_csv(RAW_FEATURE_COLUMNS_PATH, encoding="utf-8-sig")
zero_variance_df = pd.read_csv(ZERO_VARIANCE_FEATURES_PATH, encoding="utf-8-sig")
final_feature_columns_df = pd.read_csv(FINAL_FEATURE_COLUMNS_PATH, encoding="utf-8-sig")

raw_features = raw_feature_columns_df["feature"].astype(str).tolist()
removed_features = zero_variance_df["removed_feature"].astype(str).tolist()
final_features = final_feature_columns_df["feature"].astype(str).tolist()
checkpoint_features = [str(feature) for feature in checkpoint.get("feature_columns", [])]

computed_final_features = [
    feature for feature in raw_features
    if feature not in set(removed_features)
]

feature_contract_rows = []

for index, feature in enumerate(raw_features):
    feature_contract_rows.append(
        {
            "raw_feature_index": index,
            "feature": feature,
            "raw_feature_group": raw_feature_columns_df.loc[
                raw_feature_columns_df["feature"] == feature,
                "feature_group",
            ].iloc[0],
            "removed_by_zero_variance": feature in set(removed_features),
            "model_feature_index": (
                final_features.index(feature)
                if feature in final_features
                else np.nan
            ),
        }
    )

feature_contract_df = pd.DataFrame(feature_contract_rows)

corrected_alignment_checks = pd.DataFrame(
    [
        {
            "check": "raw_feature_count_matches_imputer",
            "passed": len(raw_features) == getattr(imputer, "n_features_in_", None),
            "detail": f"{len(raw_features)} vs {getattr(imputer, 'n_features_in_', None)}",
        },
        {
            "check": "raw_feature_count_matches_scaler",
            "passed": len(raw_features) == getattr(scaler, "n_features_in_", None),
            "detail": f"{len(raw_features)} vs {getattr(scaler, 'n_features_in_', None)}",
        },
        {
            "check": "computed_final_features_match_final_csv",
            "passed": computed_final_features == final_features,
            "detail": f"{len(computed_final_features)} vs {len(final_features)}",
        },
        {
            "check": "final_features_match_checkpoint_order",
            "passed": final_features == checkpoint_features,
            "detail": f"{len(final_features)} vs {len(checkpoint_features)}",
        },
        {
            "check": "removed_plus_final_equals_raw_count",
            "passed": len(removed_features) + len(final_features) == len(raw_features),
            "detail": f"{len(removed_features)} + {len(final_features)} = {len(raw_features)}",
        },
        {
            "check": "removed_features_all_in_raw",
            "passed": set(removed_features).issubset(set(raw_features)),
            "detail": f"removed={len(removed_features)}",
        },
    ]
)

inference_manifest = {
    "created_at": pd.Timestamp.now().isoformat(),
    "selected_experiment_id": SELECTED_EXPERIMENT_ID,
    "model_family": "mlp_current_day",
    "prediction_objective": "Predict good_sleep_label for an upcoming sleep episode using features available before sleep_start_datetime.",
    "prediction_cutoff": "sleep_start_datetime",
    "official_threshold": SELECTED_THRESHOLD,
    "optional_recall_priority_threshold": float(recall_policy_test_row["selected_threshold"]),
    "preprocessing_contract": {
        "raw_input_feature_count": len(raw_features),
        "model_input_feature_count": len(final_features),
        "zero_variance_removed_count": len(removed_features),
        "steps": [
            "Construct the 70 raw Stage 1 features in raw_feature_order.",
            "Apply median imputer fitted on train split to the 70 raw features.",
            "Apply StandardScaler fitted on train split to the imputed 70 features.",
            "Remove zero-variance features listed in zero_variance_removed_features.",
            "Pass the remaining 58 features, in final_model_feature_order, into the PyTorch MLP.",
            "Apply sigmoid to model logits.",
            "Classify with official_threshold unless an operating policy overrides it.",
        ],
    },
    "artifact_paths": {
        "raw_feature_columns": str(RAW_FEATURE_COLUMNS_PATH.relative_to(PROJECT_ROOT)),
        "zero_variance_removed_features": str(ZERO_VARIANCE_FEATURES_PATH.relative_to(PROJECT_ROOT)),
        "final_feature_columns": str(FINAL_FEATURE_COLUMNS_PATH.relative_to(PROJECT_ROOT)),
        "imputer": str(IMPUTER_PATH.relative_to(PROJECT_ROOT)),
        "scaler": str(SCALER_PATH.relative_to(PROJECT_ROOT)),
        "model_checkpoint": str(MODEL_PATH.relative_to(PROJECT_ROOT)),
        "selection_summary": str(SELECTION_SUMMARY_PATH.relative_to(PROJECT_ROOT)),
        "threshold_policy": str(THRESHOLD_POLICY_PATH.relative_to(PROJECT_ROOT)),
    },
    "raw_feature_order": raw_features,
    "zero_variance_removed_features": removed_features,
    "final_model_feature_order": final_features,
    "model_architecture": {
        "input_dim": checkpoint.get("input_dim"),
        "hidden_dims": list(checkpoint.get("hidden_dims")),
        "dropout": checkpoint.get("dropout"),
        "best_epoch": checkpoint.get("best_epoch"),
    },
    "performance_reference": {
        "representative_test_balanced_accuracy": float(selected_row["test_balanced_accuracy"]),
        "representative_test_roc_auc": float(selected_row["test_roc_auc"]),
        "representative_test_average_precision": float(selected_row["test_average_precision"]),
        "representative_test_precision": float(selected_row["test_precision"]),
        "representative_test_recall": float(selected_row["test_recall"]),
        "bootstrap_balanced_accuracy_ci_95": [
            float(ba_ci["ci_lower_2_5"]) if "ba_ci" in globals() else None,
            float(ba_ci["ci_upper_97_5"]) if "ba_ci" in globals() else None,
        ],
    },
    "validation_checks": corrected_alignment_checks.to_dict(orient="records"),
}

INFERENCE_MANIFEST_PATH.write_text(
    json.dumps(inference_manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
feature_contract_df.to_csv(
    INFERENCE_FEATURE_CONTRACT_PATH,
    index=False,
    encoding="utf-8-sig",
)

log_entry = f"""

## 2026-06-29 - Pre-sleep inference manifest created

Created corrected inference manifest for final pre-sleep model.

- Manifest: `{INFERENCE_MANIFEST_PATH.relative_to(PROJECT_ROOT)}`
- Feature contract: `{INFERENCE_FEATURE_CONTRACT_PATH.relative_to(PROJECT_ROOT)}`
- Raw input features: {len(raw_features)}
- Zero-variance removed features: {len(removed_features)}
- Model input features: {len(final_features)}
- Contract: raw 70 features -> train imputer -> train scaler -> remove 12 zero-variance features -> 58 model features -> MLP.
"""

with LOG_PATH.open("a", encoding="utf-8") as f:
    f.write(log_entry)

print("=== Corrected Inference Manifest Saved ===")
print("manifest:", INFERENCE_MANIFEST_PATH.relative_to(PROJECT_ROOT), INFERENCE_MANIFEST_PATH.exists())
print("feature contract:", INFERENCE_FEATURE_CONTRACT_PATH.relative_to(PROJECT_ROOT), INFERENCE_FEATURE_CONTRACT_PATH.exists())

print("\n=== Corrected Alignment Checks ===")
display(corrected_alignment_checks)

print("\n=== Feature Contract Summary ===")
display(
    feature_contract_df
    .groupby(["raw_feature_group", "removed_by_zero_variance"])
    .size()
    .reset_index(name="features")
)

print("\n=== Removed Features ===")
display(zero_variance_df)

print("\n=== Validation Checks ===")
print("failed checks:", int((~corrected_alignment_checks["passed"]).sum()))

if corrected_alignment_checks["passed"].all():
    print("corrected inference preprocessing contract passed")
else:
    print("corrected inference preprocessing contract failed; inspect before creating inference function")

print("\nlog updated:", LOG_PATH, LOG_PATH.exists())

=== Corrected Inference Manifest Saved ===
manifest: data\processed\pre_sleep_forecasting\design_c_stage1\inference_package\pre_sleep_inference_manifest.json True
feature contract: data\processed\pre_sleep_forecasting\design_c_stage1\inference_package\pre_sleep_inference_feature_contract.csv True

=== Corrected Alignment Checks ===


,check,passed,detail
0,raw_feature_count_matches_imputer,True,70 vs 70
1,raw_feature_count_matches_scaler,True,70 vs 70
2,computed_final_features_match_final_csv,True,58 vs 58
3,final_features_match_checkpoint_order,True,58 vs 58
4,removed_plus_final_equals_raw_count,True,12 + 58 = 70
5,removed_features_all_in_raw,True,removed=12



=== Feature Contract Summary ===


,raw_feature_group,removed_by_zero_variance,features
0,missing_indicator,False,25
1,missing_indicator,True,10
2,pre_sleep_intraday,False,18
3,previous_day_daily,False,9
4,previous_day_daily,True,2
5,timing,False,6



=== Removed Features ===


,removed_feature,train_scaled_std
0,previous_day_resting_hr_record_count,0.0
1,previous_day_calories_record_count,0.0
2,steps_pre_sleep_record_count_missing_ind,0.0
3,steps_pre_sleep_active_record_count_missing_ind,0.0
4,calories_pre_sleep_record_count_missing_ind,0.0
5,heart_rate_pre_sleep_record_count_missing_ind,0.0
6,pre_sleep_window_hours_missing_ind,0.0
7,sleep_start_hour_missing_ind,0.0
8,sleep_start_dayofweek_sin_missing_ind,0.0
9,sleep_start_dayofweek_cos_missing_ind,0.0



=== Validation Checks ===
failed checks: 0
corrected inference preprocessing contract passed

log updated: C:\workSpace\DeepLearnin_sleep\log\2026-06-29.md True
